# 02 — Feature Engineering Exploration

**Purpose:** Inspect the engineered feature matrix before it goes into the model — distributions, correlations, null rates, and early signs that the features separate anomalies from normal accounts.

This is one of the most important notebooks in the project. Machine learning models are only as good as the features you give them. Before training anything, we need to answer:
- Do any features have too many nulls to be useful?
- Are any features so heavily correlated that they're redundant?
- Do the injected anomaly accounts already look different from normal accounts in feature space?
- Are any features on wildly different scales that might dominate the model unfairly?

Run `src/aml_anomaly/models/train.py` (or `run_all.py`) first to generate `data/features/feature_matrix.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)

DATA_DIR = Path("../data")

In [ ]:
features = pd.read_csv(DATA_DIR / "features" / "feature_matrix.csv")
injected = pd.read_csv(DATA_DIR / "raw" / "injected_anomalies.csv")

features["is_injected"] = features["account_id"].isin(injected["account_id"])

# Separate the feature columns from metadata
meta_cols = ["account_id", "is_injected"]
feature_cols = [c for c in features.columns if c not in meta_cols]

print(f"Feature matrix: {features.shape[0]} accounts × {len(feature_cols)} features")
print(f"Injected anomaly accounts: {features['is_injected'].sum()}")

## 1. Null rates

The modeling pipeline drops features with >20% nulls and imputes the rest with the column median. Here we see what we're working with before that step.

In [ ]:
null_pct = (features[feature_cols].isnull().sum() / len(features) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) == 0:
    print("No nulls in any feature column.")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    null_pct.plot.bar(ax=ax, color="coral", edgecolor="white")
    ax.axhline(20, color="red", linestyle="--", label="20% drop threshold")
    ax.set_title("Null Rate by Feature")
    ax.set_ylabel("% null")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Features above 20% null threshold (will be dropped): {(null_pct > 20).sum()}")

## 2. Feature distributions

Looking at how each feature is distributed across all accounts. We want to see:
- Most features should be right-skewed (most accounts are normal, a few are extreme)
- The injected anomaly accounts (shown in red) should sit in the tails of the distribution for their relevant features

In [ ]:
normal_df = features[~features["is_injected"]]
anomaly_df = features[features["is_injected"]]

n_cols = 3
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.hist(normal_df[col].dropna(), bins=40, alpha=0.7, label="Normal", edgecolor="none", density=True)
    ax.hist(anomaly_df[col].dropna(), bins=20, alpha=0.85, label="Injected", color="crimson", edgecolor="none", density=True)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

axes[0].legend(fontsize=8)
fig.suptitle("Feature Distributions: Normal (blue) vs. Injected Anomaly (red)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation heatmap

Highly correlated features are redundant — they carry the same information and can distort distance-based models like LOF. PCA will handle much of this, but it's useful to see which features travel together.

A correlation close to +1.0 or -1.0 means the two features are essentially measuring the same thing.

In [ ]:
corr = features[feature_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (it's a mirror)
sns.heatmap(
    corr,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.3,
    annot=False,
    ax=ax,
)
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Strongest correlations

Pull out the top positively and negatively correlated pairs so we know exactly which features overlap.

In [ ]:
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_a", "feature_b", "correlation"]
corr_pairs = corr_pairs.reindex(corr_pairs["correlation"].abs().sort_values(ascending=False).index)

print("Top 15 strongest correlations:")
print(corr_pairs.head(15).to_string(index=False))

## 5. Summary statistics: normal vs. injected

For each feature, compare the mean value in normal accounts vs. injected anomaly accounts. A large difference confirms the feature is doing useful work for the model.

In [ ]:
normal_means = normal_df[feature_cols].mean()
anomaly_means = anomaly_df[feature_cols].mean()
normal_stds = normal_df[feature_cols].std()

separation = pd.DataFrame({
    "normal_mean": normal_means,
    "anomaly_mean": anomaly_means,
    "separation_sigma": (anomaly_means - normal_means) / (normal_stds + 1e-9),
}).sort_values("separation_sigma", key=abs, ascending=False)

print("Features ranked by separation between normal and injected accounts (in σ):")
print(separation.round(3).head(20).to_string())

## 6. Feature scale check

Before we scale the features, it helps to see how different the raw ranges are. This is why StandardScaler is essential — a feature measured in milliseconds and a feature measured in percentages would otherwise have wildly different influence on distance-based models.

In [ ]:
scale_summary = features[feature_cols].agg(["min", "max", "mean", "std"]).T
scale_summary["range"] = scale_summary["max"] - scale_summary["min"]
print(scale_summary.sort_values("range", ascending=False).round(3).to_string())